In [ ]:
import numpy as np

def GaussSemPivot(A, b):
    # Obtém a dimensão da matriz (número de linhas/equações)
    n = len(A)

    # Cria a matriz aumentada [A|b] concatenando a matriz A com o vetor b
    # O reshape(n, 1) garante que o vetor b seja tratado como uma coluna
    M = np.hstack((A, b.reshape(n, 1)))

    # --- Fase de Eliminação (Escalonamento) ---
    # k percorre cada coluna da matriz, exceto a última
    for k in range(n-1):
        # i percorre as linhas abaixo da linha pivô (k)
        for i in range(k+1, n):
            # Calcula o multiplicador (fator) para zerar o elemento M[i, k]
            fator = M[i, k] / M[k, k]

            # Atualiza a linha i subtraindo dela o fator multiplicado pela linha pivô
            # k: garante que a operação seja feita apenas do pivô em diante (otimização)
            M[i, k:] -= fator * M[k, k:]

    # --- Fase de Substituição Reversa ---
    # Cria um vetor de zeros para armazenar a solução final x
    x = np.zeros(n)

    # Percorre as linhas de baixo para cima (da última para a primeira)
    for i in range(n-1, -1, -1):
        # Calcula o valor da incógnita x[i]
        # M[i, -1] é o termo independente (vetor b atualizado)
        # np.dot subtrai a soma dos termos já conhecidos (à direita do pivô)
        x[i] = (M[i, -1] - np.dot(M[i, i+1:n], x[i+1:])) / M[i, i]

    return x

\Com certeza! Esse código implementa o método de **Eliminação de Gauss simples** (sem pivoteamento parcial), que é a base para resolver sistemas lineares do tipo $Ax = b$.

Abaixo, comento cada linha diretamente no código e, em seguida, faço uma breve explicação dos blocos para o seu relatório.

### Código Comentado

```python
import numpy as np

def GaussSemPivot(A, b):
    # Obtém a dimensão da matriz (número de linhas/equações)
    n = len(A)
    
    # Cria a matriz aumentada [A|b] concatenando a matriz A com o vetor b
    # O reshape(n, 1) garante que o vetor b seja tratado como uma coluna
    M = np.hstack((A, b.reshape(n, 1)))

    # --- Fase de Eliminação (Escalonamento) ---
    # k percorre cada coluna da matriz, exceto a última
    for k in range(n-1):
        # i percorre as linhas abaixo da linha pivô (k)
        for i in range(k+1, n):
            # Calcula o multiplicador (fator) para zerar o elemento M[i, k]
            fator = M[i, k] / M[k, k]
            
            # Atualiza a linha i subtraindo dela o fator multiplicado pela linha pivô
            # k: garante que a operação seja feita apenas do pivô em diante (otimização)
            M[i, k:] -= fator * M[k, k:]

    # --- Fase de Substituição Reversa ---
    # Cria um vetor de zeros para armazenar a solução final x
    x = np.zeros(n)
    
    # Percorre as linhas de baixo para cima (da última para a primeira)
    for i in range(n-1, -1, -1):
        # Calcula o valor da incógnita x[i]
        # M[i, -1] é o termo independente (vetor b atualizado)
        # np.dot subtrai a soma dos termos já conhecidos (à direita do pivô)
        x[i] = (M[i, -1] - np.dot(M[i, i+1:n], x[i+1:])) / M[i, i]
        
    return x
```

---

### Explicação Teórica para o Relatório

Para enriquecer seu texto, você pode dividir a explicação em três partes fundamentais:

1.  **Matriz Aumentada:** O uso do `np.hstack` é crucial para que todas as operações elementares aplicadas aos coeficientes de $A$ sejam aplicadas simultaneamente ao vetor de termos independentes $b$. Isso mantém a equivalência do sistema original.
2.  **Eliminação Progressiva:** O objetivo deste bloco é transformar a matriz $A$ em uma **matriz triangular superior**. O algoritmo utiliza o elemento $M[k, k]$ como pivô para eliminar os elementos abaixo dele.
    *   *Nota importante:* Como o código é "sem pivoteamento", ele assume que $M[k, k]$ nunca será zero. Se for zero, o código causará um erro de divisão.
3.  **Substituição Regressiva:** Com a matriz já escalonada, a última equação possui apenas uma incógnita. O código resolve essa incógnita e "sobe" para a equação anterior, substituindo os valores já encontrados (usando o `np.dot` para fazer o produtório dos termos conhecidos) até encontrar todo o vetor $x$.

**Dica de Desempenho:** O uso de `M[i, k:]` (fatiamento de arrays) é uma prática eficiente em Python/NumPy, pois evita um terceiro loop explícito, utilizando operações vetorizadas que são muito mais rápidas.

In [ ]:
import numpy as np

def fatLU(A, b):
    n = len(A)
    # Inicializa L como uma matriz identidade (1s na diagonal, 0s fora)
    L = np.eye(n)
    # Inicializa U como uma matriz de zeros
    U = np.zeros((n, n))

    # --- Fase de Decomposição ---
    for k in range(n):
        # Calcula os elementos da linha k de U
        # U[k, k:] recebe a linha de A menos o produto das linhas anteriores de L e U
        # O np.round(..., 2) arredonda o resultado para 2 casas decimais
        U[k, k:] = np.round(A[k, k:] - L[k, :k].dot(U[:k, k:]), 2)

        # Calcula os elementos da coluna k de L (abaixo da diagonal)
        # Divide a diferença pelo pivô U[k, k]
        if k < n - 1: # Evita erro de índice na última iteração
            L[k+1:, k] = np.round((A[k+1:, k] - L[k+1:, :k].dot(U[:k, k])) / U[k, k], 2)

    # --- Resolução dos Sistemas Triangulares ---

    # 1. Resolve Ly = b (Substituição Progressiva / Forward Substitution)
    y = np.zeros(n)
    for i in range(n):
        # Calcula y[i] subtraindo o produto interno dos elementos já encontrados
        y[i] = np.round((b[i] - np.dot(L[i, :i], y[:i])), 2)

    # 2. Resolve Ux = y (Substituição Regressiva / Backward Substitution)
    x = np.zeros(n)
    for i in reversed(range(n)):
        # Calcula x[i] subtraindo o produto de U pelos x já conhecidos e dividindo pelo pivô
        x[i] = np.round((y[i] - np.dot(U[i, i+1:], x[i+1:])) / U[i, i], 2)

    return L, U, x

\Este algoritmo implementa a Decomposição LU, que fatora uma matriz $A$ no produto de uma matriz triangular inferior ($L$, de Lower) e uma matriz triangular superior ($U$, de Upper). A grande vantagem aqui é que, uma vez fatorada, você resolve qualquer sistema $Ax = b$ apenas com substituições, o que é computacionalmente barato.
Este algoritmo implementa a **Decomposição LU**, que fatora uma matriz $A$ no produto de uma matriz triangular inferior ($L$, de *Lower*) e uma matriz triangular superior ($U$, de *Upper*). A grande vantagem aqui é que, uma vez fatorada, você resolve qualquer sistema $Ax = b$ apenas com substituições, o que é computacionalmente barato.

Aqui está o código detalhado e comentado:

### Código Comentado

```python
import numpy as np

def fatLU(A, b):
    n = len(A)
    # Inicializa L como uma matriz identidade (1s na diagonal, 0s fora)
    L = np.eye(n)
    # Inicializa U como uma matriz de zeros
    U = np.zeros((n, n))

    # --- Fase de Decomposição ---
    for k in range(n):
        # Calcula os elementos da linha k de U
        # U[k, k:] recebe a linha de A menos o produto das linhas anteriores de L e U
        # O np.round(..., 2) arredonda o resultado para 2 casas decimais
        U[k, k:] = np.round(A[k, k:] - L[k, :k].dot(U[:k, k:]), 2)

        # Calcula os elementos da coluna k de L (abaixo da diagonal)
        # Divide a diferença pelo pivô U[k, k]
        if k < n - 1: # Evita erro de índice na última iteração
            L[k+1:, k] = np.round((A[k+1:, k] - L[k+1:, :k].dot(U[:k, k])) / U[k, k], 2)

    # --- Resolução dos Sistemas Triangulares ---
    
    # 1. Resolve Ly = b (Substituição Progressiva / Forward Substitution)
    y = np.zeros(n)
    for i in range(n):
        # Calcula y[i] subtraindo o produto interno dos elementos já encontrados
        y[i] = np.round((b[i] - np.dot(L[i, :i], y[:i])), 2)

    # 2. Resolve Ux = y (Substituição Regressiva / Backward Substitution)
    x = np.zeros(n)
    for i in reversed(range(n)):
        # Calcula x[i] subtraindo o produto de U pelos x já conhecidos e dividindo pelo pivô
        x[i] = np.round((y[i] - np.dot(U[i, i+1:], x[i+1:])) / U[i, i], 2)

    return L, U, x
```

---

### Pontos Chave para o seu Relatório

Para a sua análise no relatório, vale destacar estes três aspectos técnicos:

*   **A Estratégia de Arredondamento:** O código utiliza `np.round(..., 2)` em todas as etapas. Isso é interessante para observar a **propagação de erros de truncamento**. Em sistemas reais, arredondar a cada passo pode afastar a solução final do valor exato, o que é um ótimo ponto de discussão sobre estabilidade numérica.
*   **Fatoração "Doolittle":** Note que `L` é inicializada com `np.eye(n)`. Isso significa que a diagonal principal de $L$ é composta por $1$s. Essa é uma convenção comum (método de Doolittle) para garantir que a decomposição seja única.
*   **A Lógica dos Dois Sistemas:** Explique que o método transforma um problema difícil ($Ax = b$) em dois problemas fáceis:
    1.  Primeiro resolve-se $Ly = b$ para achar um vetor intermediário $y$.
    2.  Depois resolve-se $Ux = y$ para achar a solução real $x$.
    *   *Por que?* Porque matrizes triangulares são resolvidas apenas com substituição direta, sem necessidade de novas eliminações.

**Nota importante:** Se no seu teste o pivô `U[k, k]` for zero, o código retornará um erro ou `inf`. Isso acontece porque esse algoritmo específico não inclui **pivoteamento parcial** (troca de linhas), que é uma melhoria usada para evitar divisões por zero ou números muito pequenos.

\A Fatoração de Cholesky é um caso especial da decomposição LU, usada quando a matriz $A$ é simétrica e positiva definida (todos os seus autovalores são positivos). Ela fatora a matriz de modo que $A = G \cdot G^T$, onde $G$ é uma matriz triangular inferior com a diagonal positiva.Aqui está o código detalhado para o seu relatório:

In [ ]:
import numpy as np

def fatCholesky(A, b):
    n = len(A)
    # Inicializa a matriz G (matriz de Cholesky) com zeros
    G = np.zeros((n, n))

    # --- Fase de Fatoração de Cholesky ---
    for i in range(n):
        for j in range(i + 1):
            # Calcula o somatório dos produtos dos elementos já calculados na linha i e j
            # Este 's' representa a parte interna da fórmula de Cholesky
            s = np.round(sum(G[i, k] * G[j, k] for k in range(j)), 2)

            if i == j:
                # Elementos da diagonal principal: raiz quadrada da diferença
                # Garante que G[i,i] * G[i,i] + soma = A[i,i]
                G[i, j] = np.round(np.sqrt(A[i, i] - s), 2)
            else:
                # Elementos fora da diagonal: divisão pela diagonal de G
                G[i, j] = np.round((A[i, j] - s) / G[j, j], 2)

    # Gera a matriz transposta G.T (G transposta)
    GT = np.round(G.T, 2)

    # --- Resolução de G · y = b (Substituição Progressiva) ---
    y = np.zeros(n)
    for i in range(n):
        # soma recebe o produto escalar da linha atual de G pelos valores de y já encontrados
        soma = np.round(np.dot(G[i, :i], y[:i]), 2)
        # Isola y[i] dividindo pelo elemento da diagonal de G
        y[i] = np.round((b[i] - soma) / G[i, i], 2)

    # --- Resolução de G.T · x = y (Substituição Regressiva) ---
    x = np.zeros(n)
    for i in reversed(range(n)):
        # soma recebe o produto escalar da linha atual de GT pelos valores de x já encontrados
        soma = np.round(np.dot(GT[i, i+1:], x[i+1:]), 2)
        # Isola x[i] dividindo pelo elemento da diagonal de GT
        x[i] = np.round((y[i] - soma) / GT[i, i], 2)

    # Exibição dos resultados para conferência
    print("Matriz G (triangular inferior):")
    print(G)
    print("\nMatriz G.T (transposta de G):")
    print(GT)
    print("\nSolução x do sistema A·x = b:")
    print(x)

    return G, GT, x

\A **Fatoração de Cholesky** é um caso especial da decomposição LU, usada quando a matriz $A$ é **simétrica** e **positiva definida** (todos os seus autovalores são positivos). Ela fatora a matriz de modo que $A = G \cdot G^T$, onde $G$ é uma matriz triangular inferior com a diagonal positiva.

Aqui está o código detalhado para o seu relatório:

### Código Comentado

```python
import numpy as np

def fatCholesky(A, b):
    n = len(A)
    # Inicializa a matriz G (matriz de Cholesky) com zeros
    G = np.zeros((n, n))

    # --- Fase de Fatoração de Cholesky ---
    for i in range(n):
        for j in range(i + 1):
            # Calcula o somatório dos produtos dos elementos já calculados na linha i e j
            # Este 's' representa a parte interna da fórmula de Cholesky
            s = np.round(sum(G[i, k] * G[j, k] for k in range(j)), 2)
            
            if i == j:
                # Elementos da diagonal principal: raiz quadrada da diferença
                # Garante que G[i,i] * G[i,i] + soma = A[i,i]
                G[i, j] = np.round(np.sqrt(A[i, i] - s), 2)
            else:
                # Elementos fora da diagonal: divisão pela diagonal de G
                G[i, j] = np.round((A[i, j] - s) / G[j, j], 2)

    # Gera a matriz transposta G.T (G transposta)
    GT = np.round(G.T, 2)

    # --- Resolução de G · y = b (Substituição Progressiva) ---
    y = np.zeros(n)
    for i in range(n):
        # soma recebe o produto escalar da linha atual de G pelos valores de y já encontrados
        soma = np.round(np.dot(G[i, :i], y[:i]), 2)
        # Isola y[i] dividindo pelo elemento da diagonal de G
        y[i] = np.round((b[i] - soma) / G[i, i], 2)

    # --- Resolução de G.T · x = y (Substituição Regressiva) ---
    x = np.zeros(n)
    for i in reversed(range(n)):
        # soma recebe o produto escalar da linha atual de GT pelos valores de x já encontrados
        soma = np.round(np.dot(GT[i, i+1:], x[i+1:]), 2)
        # Isola x[i] dividindo pelo elemento da diagonal de GT
        x[i] = np.round((y[i] - soma) / GT[i, i], 2)

    # Exibição dos resultados para conferência
    print("Matriz G (triangular inferior):")
    print(G)
    print("\nMatriz G.T (transposta de G):")
    print(GT)
    print("\nSolução x do sistema A·x = b:")
    print(x)

    return G, GT, x
```

---

### Insights para o seu Relatório

1.  **Simetria e Eficiência:** O método de Cholesky é cerca de **duas vezes mais rápido** que a fatoração LU comum, pois ele "sabe" que a matriz é simétrica e só precisa calcular metade dos elementos (a matriz $G$).
2.  **O Problema da Raiz Quadrada:** Observe a linha `np.sqrt(A[i, i] - s)`. Se o valor dentro da raiz for negativo, o algoritmo falhará (números complexos). Isso acontece se a matriz não for positiva definida. No relatório, você pode mencionar que essa é uma forma de **testar** se uma matriz é positiva definida.
3.  **Vantagem do Armazenamento:** Em aplicações reais de engenharia, como no Método dos Elementos Finitos, a fatoração de Cholesky economiza muita memória, pois você só precisa armazenar $G$ para representar todo o sistema.
4.  **Arredondamento Crítico:** Como este código utiliza `np.round(..., 2)` e `sqrt`, pequenos erros de arredondamento podem se acumular rapidamente. É um ótimo exemplo para discutir como o erro de precisão numérica se comporta em operações não lineares (raízes).


O **Método de Gauss-Jacobi** é um algoritmo **iterativo**. Diferente dos anteriores (Gauss, LU, Cholesky), que são métodos diretos, este busca a solução através de sucessivas aproximações até atingir uma precisão desejada.


In [ ]:
#Algoritmo Gauss-Jacobi - Filho

def gaussJacobi(A, b, x0, prec, kMax):
  # Inicializa o vetor x que armazenará a nova aproximação
x = np.zeros(n)
it = 0  # Inicializa o contador de iterações

# Loop principal: continua até atingir o erro (prec) ou o limite de iterações (kMax)
while it < kMax:
    it += 1  # Incrementa o contador a cada nova tentativa
    print(x) # Exibe o vetor da iteração anterior para acompanhar a evolução

    # Loop para calcular cada componente x[i] da nova aproximação
    for i in range(n):
        # Começa o cálculo isolando o termo independente b[i]
        x[i] = b[i]

        # Subtrai os termos A[i,j] * x0[j] para todos os j diferentes de i
        # Importante: usa o vetor x0 (da iteração anterior) para todos os cálculos
        for j in range(n):
            if j != i:
                x[i] -= A[i, j] * x0[j]

        # Divide pelo coeficiente da diagonal principal A[i,i] para isolar a incógnita
        x[i] /= A[i, i]

    # Verifica o critério de parada (convergência)
    # Calcula a norma infinita da diferença entre a nova aproximação e a anterior
    if np.linalg.norm(x - x0, np.inf) < prec:
        print("\n Convergiu:", np.round(x, 3))
        return x, it

    # Atualiza o vetor de aproximação antiga (x0) com os novos valores calculados (x)
    # O np.copy é essencial para não criar apenas uma referência de memória
    x0 = np.copy(x)

# Caso o loop termine por atingir o kMax sem atingir a precisão
print("\n aproximação após", kMax, "iterações.")
print("Última aproximação encontrada:", np.round(x, 3))
return x, it

\Iteração vs. Substituição: O ponto crucial do Gauss-Jacobi é que ele utiliza apenas valores da iteração anterior ($x_0$) para calcular todos os novos valores de $x$. Isso o torna fácil de paralelizar em computação de alto desempenho, mas geralmente o faz convergir mais lentamente que o método de Gauss-Seidel.Critério de Convergência (Norma Infinita): A função np.linalg.norm(..., np.inf) extrai o maior valor absoluto da diferença entre os vetores. Se até o valor que mais variou já for menor que a precisão (prec), o sistema todo é considerado estável/convergente.Condição de Convergência (Critério das Linhas): Para o seu relatório, é importante mencionar que este método só garante convergência se a matriz $A$ for estritamente diagonal dominante, ou seja:$$|A_{i,i}| > \sum_{j \neq i} |A_{i,j}|$$Se a diagonal não for "forte" o suficiente, o método pode divergir e os valores de $x$ crescerem ao infinito.Parâmetros de Entrada:x0: O "chute inicial". Quanto mais próximo da solução real, menos iterações serão necessárias.prec: A tolerância ao erro (ex: $10^{-3}$).kMax: Uma trava de segurança para evitar que o código rode eternamente caso o sistema não convirja.

O Método de Gauss-Seidel é uma evolução do Gauss-Jacobi. A principal diferença é que ele é "mais apressado" (no bom sentido): assim que ele calcula o valor de uma variável na iteração atual, ele já utiliza esse novo valor para calcular as próximas variáveis dentro da mesma iteração:

In [ ]:
import numpy as np

def gaussSeidel(A, b, x0, tol, kMax):
    n = len(b)
    # Cria uma cópia do chute inicial para começar as iterações
    x = x0.copy()
    it = 0

    # Inicia o laço que limita o número máximo de tentativas
    while it < kMax:
        # Armazena o x da iteração anterior para poder comparar o erro depois
        x_old = x.copy()

        # Percorre cada linha da matriz para atualizar as componentes de x
        for i in range(n):
            # Calcula a soma dos produtos de A por x.
            # A[i, :i] usa os valores de x JÁ ATUALIZADOS nesta iteração (novos)
            # A[i, i+1:] usa os valores de x da iteração anterior (ainda não atualizados)
            s = np.dot(A[i, :i], x[:i]) + np.dot(A[i, i+1:], x[i+1:])

            # Isola x[i] dividindo pela diagonal principal
            x[i] = (b[i] - s) / A[i, i]

        # Critério de parada: verifica se a diferença absoluta entre todos os
        # elementos de x (novo) e x_old (anterior) é menor que a tolerância (tol)
        if np.all(np.abs(x - x_old) < tol):
            print(x)
            return x, it

        # Exibe o progresso do vetor x a cada iteração
        print(x)
        it += 1

    # Executado caso o critério de tolerância não seja atingido no limite de iterações
    print("\n aproximação após", kMax, "iterações.")
    print("Última aproximação encontrada:", np.round(x, 3))
    return x, it


1.  **Velocidade de Convergência:** No relatório, destaque que o Gauss-Seidel geralmente converge **mais rápido** que o Gauss-Jacobi. Isso ocorre porque ele utiliza "informação fresca". Enquanto o Jacobi espera o fim da iteração para atualizar o vetor todo, o Seidel atualiza o componente $x_i$ e já o usa para calcular o $x_{i+1}$.
2.  **Uso de Memória:** Este método é mais eficiente em termos de memória que o Jacobi, pois você pode (teoricamente) sobrescrever os valores no mesmo vetor, sem precisar manter duas cópias completas o tempo todo (embora no código usemos `x_old` apenas para o teste de parada).
3.  **Critério de Sassenfeld:** Para garantir que o Gauss-Seidel vai convergir, não basta apenas o critério das linhas (do Jacobi). Existe um critério mais específico chamado **Critério de Sassenfeld**. Se a matriz atender a esse critério, a convergência é garantida.
4.  **Diferença no Cálculo de `s`:** Note o fatiamento `x[:i]` e `x[i+1:]`.
    *   `x[:i]` pega os valores que acabaram de ser calculados no loop `for i`.
    *   `x[i+1:]` pega os valores que ainda são da iteração passada.

Com este código e os anteriores, seu Relatório 2 está muito bem fundamentado! Precisa de mais alguma análise sobre esses algoritmos?

O Método de Newton Modificado é uma variação do Método de Newton para sistemas não lineares. A grande diferença aqui é a eficiência computacional: enquanto no Newton padrão você precisa calcular a inversa da matriz Jacobiana (ou resolver um sistema linear) em toda iteração, no Modificado você calcula a Jacobiana apenas uma vez (no início) e a reutiliza:

In [ ]:
import numpy as np

def newton_modificado(f, J, x0, prec1=0.5, prec2=0.5, max_iteracao=50):
    # Converte o chute inicial para um array do NumPy com ponto flutuante
    x = np.array(x0, dtype=float)

    # --- O "Pulo do Gato" do Método Modificado ---
    # Calcula a matriz Jacobiana J0 apenas UMA vez no ponto inicial x0
    # No Newton padrão, isso seria feito dentro do loop 'for'
    J0 = J(x)

    for k in range(max_iteracao):
        # Calcula o valor da função f (o resíduo) no ponto atual x
        fx = f(x)

        # Resolve o sistema linear J0 * s = -fx para encontrar o passo 's'
        # s é o quanto devemos deslocar x para chegar mais perto da raiz
        s = np.linalg.solve(J0, -fx)

        # Aplica arredondamento de 2 casas decimais no passo calculado
        s = np.round(s, 2)

        # Atualiza o valor de x somando o passo 's' e arredonda o resultado
        x = np.round(x + s, 2)

        # --- Critério de Parada ---
        # Verifica se o deslocamento em cada variável foi menor que as tolerâncias
        # abs(s[0]) checa a primeira variável, abs(s[1]) a segunda
        if abs(s[0]) < prec1 and abs(s[1]) < prec2:
            print(f"Convergência atingida na iteração {k+1}")
            break

    # Retorna a aproximação final da raiz
    return x

\Análise para o seu RelatórioPara fechar seu Relatório 2 com chave de ouro, aqui estão os pontos técnicos que você deve comentar sobre este algoritmo:Custo Computacional vs. Convergência:Vantagem: Calcular e decompor a matriz Jacobiana é a parte mais "pesada" do cálculo. Ao fixar J0, o algoritmo fica extremamente rápido por iteração.Desvantagem: Ao contrário do Newton original (que tem convergência quadrática), o Newton Modificado tem convergência linear. Isso significa que ele pode precisar de mais iterações para chegar no mesmo resultado, mas cada iteração é muito mais barata.Sensibilidade ao Chute Inicial: Como a Jacobiana não é atualizada, este método é ainda mais dependente de um bom x0. Se o chute inicial estiver longe da raiz, a matriz J0 pode não ser uma boa representação da inclinação da função lá na frente, e o método pode divergir.Critério de Parada Local: O código usa abs(s[0]) e abs(s[1]). Isso é uma verificação componente a componente. Em relatórios acadêmicos, costuma-se comparar isso com a Norma Infinita do passo $s$ (o maior valor absoluto entre eles).Uso do np.linalg.solve: É sempre mais estável numericamente usar solve(J, -fx) do que calcular a inversa explicitamente ($J^{-1} \cdot -fx$), pois a inversão de matrizes é sujeita a maiores erros de precisão.